# 🎬 Wan 2.2 A14B (4-bit) — I2V · שחזור קטע 0:57-1:01 מ-lana82 · v2.1
יעד: פריים עוגן אמיתי מהסרטון + תנועה כבדה דומה · אנכי 832x480 → אפסקייל בהמשך

In [ ]:
!pip -q install diffusers transformers accelerate safetensors sentencepiece einops bitsandbytes
import torch
assert torch.cuda.is_available(), 'אין GPU! הפעלו: זמן ריצה ← שנה ← T4'
print('✔ GPU:', torch.cuda.get_device_name(0))

In [ ]:
IMAGE_URL = 'https://aidoor.shimi.land/gallery/lana-ref/anchor_57s_full.jpg'
PROMPT = '''The woman from the reference image moving dynamically with natural fluid body motion,
dancing gracefully, hips swaying, hands flowing through the air, turning slowly,
camera slowly orbiting around her, soft cinematic lighting, photorealistic, smooth continuous motion.'''
NEGATIVE = 'blurry, low quality, distorted face, deformed hands, extra limbs, watermark, text, jittery flickering'
SEED = 57101
FRAMES = 81; FPS = 16; HEIGHT, WIDTH = 832, 480   # אנכי — כמו lana82 (1080x1600)
STEPS = 20   # אפשר 12 למהירות


## טעינת המודל בכימות 4-bit (~10 דקות, פעם אחת)
ה-transformer (14B MoE) והטקסט-אנקודר (UMT5-XXL) שניהם ב-4-bit — כך נכנסים ל-12.7GB הזיכרון של Colab.

In [ ]:
import torch
from diffusers import WanImageToVideoPipeline, WanTransformer3DModel, AutoencoderKLWan
from diffusers import UniPCMultistepScheduler, BitsAndBytesConfig as BnB4Diff
from transformers import UMT5EncoderModel, T5TokenizerFast, BitsAndBytesConfig as BnB4HF
from diffusers.utils import export_to_video, load_image

MODEL = 'Wan-AI/Wan2.2-I2V-A14B-Diffusers'
q4 = BnB4Diff(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
q4hf = BnB4HF(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)

transformer = WanTransformer3DModel.from_pretrained(MODEL, subfolder='transformer', quantization_config=q4)
text_encoder = UMT5EncoderModel.from_pretrained(MODEL, subfolder='text_encoder', quantization_config=q4hf)
tokenizer = T5TokenizerFast.from_pretrained(MODEL, subfolder='tokenizer')
vae = AutoencoderKLWan.from_pretrained(MODEL, subfolder='vae', torch_dtype=torch.float16)
scheduler = UniPCMultistepScheduler.from_pretrained(MODEL, subfolder='scheduler')

pipe = WanImageToVideoPipeline(tokenizer=tokenizer, text_encoder=text_encoder, vae=vae,
                               scheduler=scheduler, image_processor=None, image_encoder=None,
                               transformer=transformer)
pipe.enable_model_cpu_offload()
pipe.vae.enable_tiling()
print('✔ המנוע נטען ב-4-bit')

## רינדור — אפשר להתנתק, Colab ממשיך (אל תסגרו את הכרטיסייה דרך 'נהל סשן')

In [ ]:
try:
    img = load_image(IMAGE_URL)
    g = torch.Generator('cuda').manual_seed(SEED)
    out = pipe(image=img, prompt=PROMPT, negative_prompt=NEGATIVE,
               height=HEIGHT, width=WIDTH, num_frames=FRAMES,
               num_inference_steps=STEPS, guidance_scale=5.0, generator=g).frames[0]
    export_to_video(out, 'wan_output.mp4', fps=FPS)
    print('✔ נשמר: wan_output.mp4')
except torch.cuda.OutOfMemoryError:
    print('אזל הזיכרון. עצירת זמן ריצה ← הפעלה מחדש ← הורידו: FRAMES=49, STEPS=12, HEIGHT,WIDTH=480,480')
    raise

In [ ]:
from IPython.display import HTML
HTML('<video width=640 controls loop src="wan_output.mp4"></video>')

In [ ]:
import requests
r = requests.post('https://catbox.moe/user/api.php',
    data={'reqtype':'fileupload'}, files={'fileToUpload': open('wan_output.mp4','rb')})
print('🔗 שלחו את הקישור לשימי פיתוח לאכלוס הגלריה:', r.text if r.ok else 'נכשל: '+r.text)